In [ ]:
# Fix for Windows WinError 127 / fbgemm.dll DLL conflict:
# Conda ships a stub libiomp5md.dll in Library\bin that shadows PyTorch's real
# OpenMP runtime. Adding torch's lib dir first ensures Windows loads the correct one.
# import os, sys
# _torch_lib = os.path.join(sys.prefix, "Lib", "site-packages", "torch", "lib")
# if os.path.isdir(_torch_lib):
#     os.add_dll_directory(_torch_lib)


# Admissions Essay Discourse Analysis

**Four analyses, all cross-referenced against reviewer scores:**

| # | Analysis | Core technique |
|---|---|---|
| 1 | **Discourse Mapping** | Sentence-level role labeling → sequence clustering |
| 2 | **Narrative Arc** | Sentiment valence over document length → arc clustering |
| 3 | **Argument Mining** | Claim / evidence density detection |
| 4 | **Coherence** | Sentence-embedding cosine similarity chain |

**Expected CSV** (`essay_assessment/data/essays.csv`): `student_id`, `essay`, `score`

## Dependencies

In [ ]:
# Uncomment and run once if packages are missing
# !pip install transformers torch sentence-transformers \
#             scikit-learn pandas matplotlib seaborn spacy
#!python -m spacy download en_core_web_sm
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
import torch
print(torch.cuda.is_available())   # should be True
print(torch.cuda.get_device_name(0))

## Imports & config

In [ ]:
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import pandas as pd
import seaborn as sns
import spacy
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from transformers import pipeline

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

SAMPLE_N = None   # use a subset while developing; set None for a full 15k run
RANDOM_STATE = 42


In [ ]:

# ── Dataset selection — change this one variable to switch corpora ─────────────
# 'real'      → real student essays (data/Essays_df.xlsx)
# 'synthetic' → ../synthetic_data/synthetic_essays.csv
# 'freeform'  → ../synthetic_data/freeform_essays.csv
DATASET = 'real'

_ds = DATASET
DATA_SOURCES = {
    'real':      Path('../data/real/essays.xlsx'),
    'synthetic': Path('../data/synthetic/synthetic_essays.csv'),
    'freeform':  Path('../data/freeform/freeform_essays.csv'),
}
CSV_PATH = DATA_SOURCES[_ds]

# All checkpoints go in one subfolder, namespaced by dataset — runs never collide.
CHECKPOINT_DIR = Path(f'../outputs/discourse/{_ds}/checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH         = CHECKPOINT_DIR / 'discourse_checkpoint.csv'
EMOTION_CHECKPOINT_PATH = CHECKPOINT_DIR / 'emotion_checkpoint_samlowe.csv'
ARG_CHECKPOINT_PATH     = CHECKPOINT_DIR / 'argument_checkpoint.csv'
COH_CHECKPOINT_PATH     = CHECKPOINT_DIR / 'coherence_checkpoint.csv'
OUTPUT_DIR              = Path(f'../outputs/discourse/{_ds}')

print(f"Dataset : {DATASET}")
print(f"  Input      : {CSV_PATH}")
print(f"  Checkpoints: {CHECKPOINT_DIR}/")
print(f"  Outputs    : {OUTPUT_DIR}/")


## Load essays

In [ ]:
def load_essays(csv_path=CSV_PATH, sample_n=SAMPLE_N):
    p = Path(csv_path)
    if p.exists():
        if p.suffix.lower() in ('.xlsx', '.xls'):
            df = pd.read_excel(p)
            df["essay"] = df["Essay Response"]
            df["score"] = df["Essay Score"]
            df = df.dropna(subset=["essay", "score"])
        else:
            df = pd.read_csv(p)
            # synthetic / freeform CSVs already have 'essay'; score may be absent
            if 'essay' not in df.columns:
                raise ValueError(f"CSV {p.name} has no 'essay' column. Columns: {list(df.columns)}")
            if 'score' not in df.columns:
                if 'proxy_score' in df.columns:
                    df['score'] = df['proxy_score']
                    print("No 'score' column — using 'proxy_score' as stand-in.")
                else:
                    df['score'] = float('nan')
        df = df.dropna(subset=['essay'])
        if 'student_id' not in df.columns:
            df["student_id"] = df.index
        missing = [c for c in ('essay', 'score') if c not in df.columns]
        if missing:
            raise ValueError(f'Missing columns: {missing}')
        print(f'Loaded {len(df):,} essays from {p}')
    else:
        print('CSV not found — using dummy dataset.')
        df = pd.DataFrame({
            'student_id': range(1, 9),
            'score':      [3.5, 4.0, 3.0, 4.5, 3.5, 4.0, 3.0, 4.5],
            'essay': [
                'Growing up in a small town, I never imagined standing before a crowd. '
                'I struggled with anxiety for years. But joining debate changed me. '
                'I learned my voice matters. Now I want to study psychology to help others find theirs.',
                'My mother works two jobs. Every morning I watch her leave before sunrise. '
                'That image drives everything I do. I maintain a 4.0 GPA not for myself, but for her. '
                'I intend to become an engineer and give her the life she deserves.',
                'I have always believed science can solve humanity\'s greatest problems. '
                'Studies show renewable energy can reduce emissions by up to 70%. '
                'I conducted my own solar panel efficiency research. '
                'The data confirmed innovation, not incremental change, is required.',
                'Some say art is a luxury. I disagree. After Hurricane Maria devastated my community, '
                'art became our therapy. I organized murals, open mics, poetry slams. '
                'Counter to those who dismiss the humanities, I argue they are essential infrastructure.',
                'I want to be a doctor. I shadowed a cardiologist for 80 hours last summer. '
                'I observed three open-heart surgeries. Every case reinforced my commitment. '
                'Medicine is not just a career — it is a calling I cannot ignore.',
                'Leadership is not about authority; it is about service. '
                'As student council president I launched a mental health awareness week. '
                'Attendance exceeded 600 students. I am proud of that impact.',
                'When I was eight my grandmother taught me to cook. '
                'Those afternoons shaped my identity more than any classroom. '
                'Food is culture, memory, and love. I plan to study nutrition science through that lens.',
                'The first time I coded a working program, I felt a joy I had never experienced. '
                'I built a tutoring app used by 300 students in my district. '
                'Technology, I realized, is the most scalable form of empathy.',
            ],
        })

    if sample_n and len(df) > sample_n:
        df = df.sample(sample_n, random_state=RANDOM_STATE).reset_index(drop=True)
        print(f'Sampled {sample_n} essays for development run.')

    return df

df = load_essays()
df[['student_id', 'score', 'essay']].head()


## Sentence segmentation utility

In [ ]:
nlp = spacy.load('en_core_web_sm', disable=['ner', 'lemmatizer'])

def split_sentences(text):
    """Return list of sentence strings for one essay."""
    doc = nlp(str(text))
    return [s.text.strip() for s in doc.sents if s.text.strip()]

# Preview
sample_sents = split_sentences(df['essay'].iloc[0])
for i, s in enumerate(sample_sents):
    print(f'[{i}] {s}')

---
## Section 1: Discourse Mapping

Label each sentence with one of seven discourse roles using the fine-tuned RoBERTa classifier:
`Lead` · `Position` · `Claim` · `Evidence` · `Counterclaim` · `Rebuttal` · `Concluding Statement`

Then represent each essay as a **role sequence** (e.g. `L-P-C-E-E-CS`) and cluster
essays that share similar argumentative structures.

> **Model:** `roberta-base` fine-tuned on PERSUADE 2.0 — supervised 7-class sentence
> classifier.


In [ ]:
from transformers import pipeline

DISCOURSE_LABELS = [
    'Lead', 'Position', 'Claim', 'Evidence',
    'Counterclaim', 'Rebuttal', 'Concluding Statement',
]

# v2: trained with tri-span context windows + focal loss + essay-level split
# v1 (./roberta-discourse-finetuned): trained on isolated spans with class weights
FINETUNED_MODEL = './roberta-discourse-finetuned-v2'

discourse_classifier = pipeline(
    'text-classification',
    model=FINETUNED_MODEL,
    tokenizer=FINETUNED_MODEL,
    device=0,
    truncation=True,
    max_length=256,
)


In [ ]:
def label_sentences(sentences, classifier, batch_size=32):
    """
    Return predicted discourse role label for each sentence.

    The v2 model was trained on tri-span context windows:
        [prev span] </s> [target span] </s> [next span]

    We replicate the same format at inference time so the model sees the same
    structure it was trained on. Boundary sentences (first and last) get a
    one-sided window, matching the training-time behaviour for essay boundaries.
    """
    if not sentences:
        return []

    sep = classifier.tokenizer.sep_token   # '</s>' for RoBERTa

    context_inputs = []
    for i, sent in enumerate(sentences):
        prev = sentences[i - 1] if i > 0 else ''
        nxt  = sentences[i + 1] if i < len(sentences) - 1 else ''
        parts = [p for p in [prev, sent, nxt] if p]
        context_inputs.append(f' {sep} '.join(parts))

    results = classifier(context_inputs, batch_size=batch_size,
                         truncation=True, max_length=256)
    if isinstance(results, dict):
        results = [results]
    return [r['label'] for r in results]

# Demo on first essay
demo_sents  = split_sentences(df['essay'].iloc[0])
demo_labels = label_sentences(demo_sents, discourse_classifier)
for sent, lbl in zip(demo_sents, demo_labels):
    print(f'[{lbl:22s}] {sent}')


In [ ]:
from tqdm.auto import tqdm
import json

LABEL_ABBREV = {
    'Lead': 'L', 'Position': 'P', 'Claim': 'C',
    'Evidence': 'E', 'Counterclaim': 'CC', 'Rebuttal': 'R',
    'Concluding Statement': 'CS',
}

CHUNK_SIZE = 500  # essays per checkpoint save

# Resume from checkpoint if it exists
if CHECKPOINT_PATH.exists():
    saved = pd.read_csv(CHECKPOINT_PATH)
    done_ids = set(saved['student_id'])
    discourse_rows = saved.to_dict('records')
    print(f'Resuming — {len(done_ids):,} essays already processed.')
else:
    done_ids = set()
    discourse_rows = []

remaining = df[~df['student_id'].isin(done_ids)].reset_index(drop=True)

SEP = discourse_classifier.tokenizer.sep_token  # '</s>' for RoBERTa

for chunk_start in tqdm(range(0, len(remaining), CHUNK_SIZE), desc='Discourse mapping'):
    chunk = remaining.iloc[chunk_start:chunk_start + CHUNK_SIZE]

    # Step 1: split sentences per essay
    chunk_sents = [split_sentences(row['essay']) for _, row in chunk.iterrows()]

    # Step 2: build context windows per essay (essay-boundary-aware).
    # Flattening all essays first and calling label_sentences() on the flat list
    # would bleed context across essay boundaries — the last sentence of essay N
    # would incorrectly receive the first sentence of essay N+1 as "next" context.
    # Instead, we build context strings here and flatten only after.
    flat_contexts = []
    boundaries    = []   # (start, end) index into flat_contexts for each essay
    for sents in chunk_sents:
        start = len(flat_contexts)
        for i, sent in enumerate(sents):
            prev  = sents[i - 1] if i > 0 else ''
            nxt   = sents[i + 1] if i < len(sents) - 1 else ''
            parts = [p for p in [prev, sent, nxt] if p]
            flat_contexts.append(f' {SEP} '.join(parts))
        boundaries.append((start, len(flat_contexts)))

    # Step 3: single batched classifier pass over all context strings
    if flat_contexts:
        results = discourse_classifier(flat_contexts, batch_size=32,
                                       truncation=True, max_length=256)
        if isinstance(results, dict):
            results = [results]
        flat_labels = [r['label'] for r in results]
    else:
        flat_labels = []

    # Step 4: reconstruct per-essay records
    for (_, row), sents, (start, end) in zip(chunk.iterrows(), chunk_sents, boundaries):
        labels  = flat_labels[start:end]
        seq     = '-'.join(LABEL_ABBREV.get(l, '?') for l in labels)
        counts  = {lbl: labels.count(lbl) for lbl in DISCOURSE_LABELS}

        # Store per-sentence labels as JSON so Section 3 argument mining can
        # reuse them instead of re-running Stage 2 (Claim/Evidence detection).
        sent_labels_json = json.dumps(
            [{'sentence': s, 'label': l} for s, l in zip(sents, labels)]
        )
        discourse_rows.append({
            'student_id':        row['student_id'],
            'sequence':          seq,
            'n_sentences':       len(sents),
            'sent_labels':       sent_labels_json,
            **counts,
        })

    pd.DataFrame(discourse_rows).to_csv(CHECKPOINT_PATH, index=False)

discourse_df = pd.DataFrame(discourse_rows)
df = df.merge(discourse_df, on='student_id')
discourse_df.head()


In [ ]:
# Cluster essays by role count vector
role_features      = df[DISCOURSE_LABELS].values.astype(float)
role_features_norm = normalize(role_features)   # L2-normalise so length doesn't dominate

N_DISCOURSE_CLUSTERS = 4
km_discourse = KMeans(n_clusters=N_DISCOURSE_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
df['discourse_cluster'] = km_discourse.fit_predict(role_features_norm)

cluster_profiles = df.groupby('discourse_cluster')[DISCOURSE_LABELS].mean().round(2)
cluster_profiles['dominant_role'] = cluster_profiles.idxmax(axis=1)
print(cluster_profiles)

In [ ]:
from sklearn.metrics import silhouette_score

inertias, silhouettes = [], []
K_RANGE = range(2, 11)

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(role_features_norm)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(role_features_norm, labels, sample_size=2000))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K_RANGE, inertias, marker='o')
axes[0].set_title('Elbow — inertia vs k')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia')

axes[1].plot(K_RANGE, silhouettes, marker='o', color='green')
axes[1].set_title('Silhouette score vs k')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette')
plt.tight_layout(); plt.show()

In [ ]:
# Heat-map: mean role counts per cluster
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(cluster_profiles[DISCOURSE_LABELS], annot=True, fmt='.1f',
            cmap='YlOrRd', ax=ax)
ax.set_title('Mean discourse role counts by cluster')
ax.set_xlabel('Discourse role')
ax.set_ylabel('Cluster')
plt.tight_layout()
plt.show()

# Score by cluster
score_disc = df.groupby('discourse_cluster')['score'].agg(['mean', 'std', 'count']).round(3)
score_disc.columns = ['mean_score', 'std_score', 'n_essays']
score_disc = score_disc.join(cluster_profiles[['dominant_role']])
print(score_disc)

---
## Section 2: Narrative Arcs

Score each sentence's emotional valence using `SamLowe/roberta-base-go_emotions`
(28-class GoEmotions → mapped to a scalar valence). Normalise each essay's arc to
a fixed-length vector (10 buckets), then cluster into narrative types:

| Cluster name | Shape | Reagan et al. analogue |
|---|---|---|
| **The Comeback** | Starts negative, ends clearly positive | "Man in a Hole" (fall-rise) |
| **The Builder** | Positive throughout, rising toward the end | "Rags to Riches" (steady rise) |
| **The Fall** | Starts positive, ends clearly negative | "Tragedy" (steady fall) |
| **The Tumble** | Already negative, declining further | *(extended fall)* |
| **The Valley** | Dips in the middle, recovers by the close | "Cinderella" variant |
| **The Peak** | Rises to a high point then falls | "Icarus" (rise-fall) |
| **Steady Success** | Flat and consistently positive throughout | *(flat positive)* |
| **Dark / Flat** | Flat and consistently negative or neutral | *(flat negative)* |
| **Neutral / Mixed** | No dominant directional pattern | *(ambiguous)* |

**Checkpoint:** `emotion_checkpoint_samlowe.csv`  
**Previous model:** `j-hartmann/emotion-english-distilroberta-base` (7-class) — checkpoint saved as `emotion_checkpoint.csv`

> **References:**
> - Reagan, A., Mitchell, L., Kiley, D., Danforth, C. M., & Dodds, P. S. (2016). The emotional arcs of stories are dominated by six basic shapes. *EPJ Data Science*, 5(1), 31. https://doi.org/10.1140/epjds/s13688-016-0093-1
> - Vonnegut, K. (1981). *Palm Sunday: An Autobiographical Collage*. Delacorte Press. (Original "shapes of stories" concept.)
> - Jockers, M. (2015). Revealing Sentiment and Plot Arcs with the Syuzhet Package. http://www.matthewjockers.net/2015/02/02/syuzhet/
>
> *Arc names used here ("The Comeback," "The Builder," etc.) are descriptive labels adapted from Vonnegut's original named shapes and Reagan et al.'s neutral shape descriptors.*


In [ ]:

from tqdm import tqdm

# ── Model selection ───────────────────────────────────────────────────────────
# Previous model (7-class); checkpoint saved on disk as emotion_checkpoint.csv.
# To revert, comment out the SamLowe block below and uncomment this block.
#
# MODEL_EMOTION = 'j-hartmann/emotion-english-distilroberta-base'
# EMOTION_VALENCE = {
#     'joy':      1.0,
#     'surprise': 0.3,
#     'neutral':  0.0,
#     'fear':    -0.5,
#     'sadness': -0.7,
#     'anger':   -0.8,
#     'disgust': -0.9,
# }
# EMOTION_CHECKPOINT_PATH = Path('emotion_checkpoint.csv')  # <- hartmann checkpoint

# Current model: SamLowe/roberta-base-go_emotions — 28-class GoEmotions.
# Richer vocabulary → more nuanced valence signal for personal narrative essays.
MODEL_EMOTION = 'SamLowe/roberta-base-go_emotions'
EMOTION_VALENCE = {
    # ── High positive ──────────────────────────────────────────────────────────
    'admiration':     0.9,
    'amusement':      0.8,
    'approval':       0.7,
    'excitement':     0.9,
    'gratitude':      0.9,
    'joy':            1.0,
    'love':           1.0,
    'optimism':       0.8,
    'pride':          0.8,
    'relief':         0.6,
    # ── Mild positive / ambivalent ─────────────────────────────────────────────
    'caring':         0.5,
    'curiosity':      0.4,
    'desire':         0.4,
    'realization':    0.2,
    'surprise':       0.1,
    # ── Neutral ───────────────────────────────────────────────────────────────
    'neutral':        0.0,
    'confusion':      0.0,
    # ── Mild negative ─────────────────────────────────────────────────────────
    'annoyance':     -0.4,
    'disappointment':-0.5,
    'disapproval':   -0.5,
    'embarrassment': -0.5,
    'nervousness':   -0.4,
    # ── Strong negative ───────────────────────────────────────────────────────
    'anger':         -0.8,
    'disgust':       -0.8,
    'fear':          -0.6,
    'grief':         -0.9,
    'remorse':       -0.7,
    'sadness':       -0.7,
}

emotion_classifier = pipeline(
    'text-classification',
    model=MODEL_EMOTION,
    device=0,
    truncation=True,
    max_length=512,
)

ARC_BUCKETS       = 10    # normalised arc length for comparison across essays
ARC_BATCH_SZ      = 64    # sentences per GPU batch — increase if VRAM allows
ARC_COLS          = [f'arc_{i}' for i in range(ARC_BUCKETS)]
EMOTION_CHUNK_SIZE = 500   # essays per checkpoint save

def _interp_arc(scores, n_buckets=ARC_BUCKETS):
    """Resample a per-sentence valence list to a fixed-length arc."""
    if not scores:
        return [0.0] * n_buckets
    x_orig = np.linspace(0, 1, len(scores))
    x_new  = np.linspace(0, 1, n_buckets)
    return list(np.interp(x_new, x_orig, scores))

# ── Resume from checkpoint if it exists ──────────────────────────────────────
if EMOTION_CHECKPOINT_PATH.exists():
    saved_emotion = pd.read_csv(EMOTION_CHECKPOINT_PATH)
    done_emotion_ids = set(saved_emotion['student_id'])
    arc_rows = saved_emotion.to_dict('records')
    print(f'Resuming — {len(done_emotion_ids):,} essays already processed.')
else:
    done_emotion_ids = set()
    arc_rows = []

remaining_emotion = df[~df['student_id'].isin(done_emotion_ids)].reset_index(drop=True)

# ── Process remaining essays in chunks, saving every EMOTION_CHUNK_SIZE essays ─
for chunk_start in tqdm(range(0, len(remaining_emotion), EMOTION_CHUNK_SIZE), desc='Emotion arcs'):
    chunk = remaining_emotion.iloc[chunk_start:chunk_start + EMOTION_CHUNK_SIZE]

    # Step 1: split sentences for this chunk
    chunk_sents = [split_sentences(essay) for essay in chunk['essay']]

    # Step 2: flatten and run one batched GPU pass
    flat_sents = [s for sents in chunk_sents for s in sents]
    flat_results = []
    for i in range(0, len(flat_sents), ARC_BATCH_SZ):
        batch = flat_sents[i : i + ARC_BATCH_SZ]
        flat_results.extend(emotion_classifier(batch, batch_size=ARC_BATCH_SZ))

    # Step 3: reconstruct per-essay arcs and append to arc_rows
    cursor = 0
    for (_, row), sents in zip(chunk.iterrows(), chunk_sents):
        results = flat_results[cursor:cursor + len(sents)]
        cursor += len(sents)
        scores = [EMOTION_VALENCE.get(r['label'], 0.0) for r in results]
        arc = _interp_arc(scores)
        arc_rows.append({
            'student_id': row['student_id'],
            **{f'arc_{i}': v for i, v in enumerate(arc)},
        })

    # Step 4: save checkpoint
    pd.DataFrame(arc_rows).to_csv(EMOTION_CHECKPOINT_PATH, index=False)
    print(f'  Checkpoint saved — {len(arc_rows):,} essays total.')

# ── Merge arc columns back into df ────────────────────────────────────────────
arc_df = pd.DataFrame(arc_rows)
df = df.drop(columns=[c for c in ARC_COLS if c in df.columns], errors='ignore')
df = df.merge(arc_df, on='student_id')
arc_matrix = df[ARC_COLS].values   # shape: (n_essays, ARC_BUCKETS)

# Preview arcs for first 3 essays
fig, axes = plt.subplots(1, 3, figsize=(13, 3), sharey=True)
for ax, (_, row) in zip(axes, df.head(3).iterrows()):
    arc = [row[f'arc_{i}'] for i in range(ARC_BUCKETS)]
    ax.plot(arc, marker='o', linewidth=2)
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
    ax.set_ylim(-1.1, 1.1)
    ax.set_title(f"ID {row['student_id']}")
    ax.set_xlabel('Position (normalised)')
axes[0].set_ylabel('Emotion valence')
plt.suptitle(f"Emotion-based valence arcs — {MODEL_EMOTION.split('/')[-1]}")
plt.tight_layout()
plt.show()


In [ ]:

from sklearn.metrics import silhouette_score

inertias_arc, silhouettes_arc = [], []
K_RANGE_ARC = range(2, 11)

for k in K_RANGE_ARC:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(arc_matrix)
    inertias_arc.append(km.inertia_)
    silhouettes_arc.append(silhouette_score(arc_matrix, labels, sample_size=2000))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K_RANGE_ARC, inertias_arc, marker='o')
axes[0].set_title('Elbow — inertia vs k (narrative arcs)')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia')

axes[1].plot(K_RANGE_ARC, silhouettes_arc, marker='o', color='green')
axes[1].set_title('Silhouette score vs k (narrative arcs)')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette')
plt.tight_layout(); plt.show()


In [ ]:
N_ARC_CLUSTERS = 3   # chosen by elbow + silhouette charts above
km_arc = KMeans(n_clusters=N_ARC_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
df['arc_cluster'] = km_arc.fit_predict(arc_matrix)

arc_cols     = [f'arc_{i}' for i in range(ARC_BUCKETS)]
cluster_arcs = df.groupby('arc_cluster')[arc_cols].mean()

def arc_name(row):
    """
    Name a narrative arc based on its valence trajectory.
    Works on both a single essay's arc and a cluster mean arc.

    Patterns tested in order of specificity:
      The Comeback    — starts negative, ends clearly positive
      The Builder     — starts positive, rises further
      The Fall        — starts positive, ends clearly negative
      The Tumble      — already negative, declining further
      The Valley      — mid-essay dip below both endpoints
      The Peak        — mid-essay surge above both endpoints
      Steady Success  — flat and consistently positive
      Dark / Flat     — flat and consistently negative/neutral
      Neutral / Mixed — no strong directional pattern
    """
    vals  = np.array(row) if not hasattr(row, 'values') else row.values
    start = vals[0]
    end   = vals[-1]
    mid   = vals[len(vals) // 2]
    mean_ = vals.mean()
    delta = end - start

    if delta > 0.25 and start < 0:
        return 'The Comeback'
    if delta > 0.15 and start >= 0:
        return 'The Builder'
    if delta < -0.25 and start > 0:
        return 'The Fall'
    if delta < -0.15 and start <= 0:
        return 'The Tumble'
    if mid < min(start, end) - 0.15:
        return 'The Valley'
    if mid > max(start, end) + 0.15:
        return 'The Peak'
    if mean_ > 0.25:
        return 'Steady Success'
    if mean_ < -0.15:
        return 'Dark / Flat'
    return 'Neutral / Mixed'

# ── Cluster-based label (denoised — applied to cluster mean arc) ───────────────
# Each essay in a cluster gets the label of its cluster's mean trajectory.
arc_names = {
    cid: f"{arc_name(row)} (cluster {cid})"
    for cid, row in cluster_arcs.iterrows()
}
df['arc_label'] = df['arc_cluster'].map(arc_names)

# ── Rule-based label (per-essay — applied to each individual arc) ──────────────
# Noisier, but directly interpretable and not dependent on cluster membership.
df['arc_label_rule'] = df[arc_cols].apply(arc_name, axis=1)

# ── Agreement analysis ─────────────────────────────────────────────────────────
# Strip the cluster number suffix for comparison (we only care about the name).
df['_arc_label_base'] = df['arc_label'].str.replace(r'\s*\(cluster \d+\)', '', regex=True)
agreement = (df['_arc_label_base'] == df['arc_label_rule']).mean()
print(f"Cluster label vs. rule-based label agreement: {agreement:.1%}\n")

# Cross-tab: what shape does the rule call essays in each cluster?
print("Rule-based label distribution within each cluster:")
crosstab = pd.crosstab(df['arc_label'], df['arc_label_rule'], margins=True)
print(crosstab)
df.drop(columns=['_arc_label_base'], inplace=True)

# ── Cluster mean trajectory plot ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
colors = cm.tab10(np.linspace(0, 0.9, N_ARC_CLUSTERS))
for (cluster_id, row), color in zip(cluster_arcs.iterrows(), colors):
    n = (df['arc_cluster'] == cluster_id).sum()
    ax.plot(row.values, marker='o', linewidth=2.5, color=color,
            label=f"{arc_names[cluster_id]} (n={n})")
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_xticks(range(ARC_BUCKETS))
ax.set_xticklabels([f'{int(i/ARC_BUCKETS*100)}%' for i in range(ARC_BUCKETS)])
ax.set_xlabel('Essay position')
ax.set_ylabel('Mean sentiment valence')
ax.set_title('Narrative arc clusters — mean trajectories (cluster-based labels)')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

# ── Rule-based label distribution ─────────────────────────────────────────────
print("\nRule-based label counts (per-essay):")
print(df['arc_label_rule'].value_counts())

# Score by arc type — both versions
print("\nMean score by cluster label:")
print(df.groupby('arc_label')['score'].agg(['mean', 'std', 'count']).round(3))
print("\nMean score by rule-based label:")
print(df.groupby('arc_label_rule')['score'].agg(['mean', 'std', 'count']).round(3))


In [ ]:

from scipy import stats

# ── Score distribution by narrative arc cluster ───────────────────────────────
arc_order = sorted(df['arc_label'].unique())
groups    = [df.loc[df['arc_label'] == lbl, 'score'].dropna() for lbl in arc_order]

# One-way ANOVA + Kruskal-Wallis (non-parametric, better for ordinal scores)
f_stat, p_anova   = stats.f_oneway(*groups)
h_stat, p_kruskal = stats.kruskal(*groups)

# Epsilon-squared effect size for Kruskal-Wallis
# ε² ≈ 0.01 negligible, 0.06 small, 0.14 medium, 0.26 large
k = len(groups)
n = sum(len(g) for g in groups)
epsilon_sq = (h_stat - k + 1) / (n - k)

print(f"One-way ANOVA:   F = {f_stat:.3f},  p = {p_anova:.4f}")
print(f"Kruskal-Wallis:  H = {h_stat:.3f},  p = {p_kruskal:.4f}")
print(f"Effect size:     ε² = {epsilon_sq:.4f}  ({'negligible' if epsilon_sq < 0.01 else 'small' if epsilon_sq < 0.06 else 'medium' if epsilon_sq < 0.14 else 'large'})")
print(f"  → With n={n:,}, even trivial differences reach significance. ε² is what matters.")
print()

# Summary table
arc_score_summary = (
    df.groupby('arc_label')['score']
      .agg(['mean', 'std', 'median', 'count'])
      .rename(columns={'mean': 'mean_score', 'std': 'std_score',
                       'median': 'median_score', 'count': 'n_essays'})
      .round(3)
)
print(arc_score_summary.sort_values('mean_score', ascending=False))

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: box plot
ax = axes[0]
data_for_box = [g.values for g in groups]
bp = ax.boxplot(data_for_box, labels=arc_order, patch_artist=True)
colors_box = cm.tab10(np.linspace(0, 0.5, len(arc_order)))
for patch, c in zip(bp['boxes'], colors_box):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)
ax.set_xlabel('Narrative arc cluster')
ax.set_ylabel('Score')
ax.set_title('Score distribution by narrative arc')
ax.tick_params(axis='x', rotation=20)

# Right: mean ± 1 SD bar chart
ax2 = axes[1]
means = [g.mean() for g in groups]
stds  = [g.std()  for g in groups]
x_pos = np.arange(len(arc_order))
ax2.bar(x_pos, means, yerr=stds, capsize=5,
        color=colors_box, alpha=0.8, edgecolor='grey')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(arc_order, rotation=20)
ax2.set_ylabel('Mean score')
ax2.set_title(f'Mean score ± SD by arc  (ε² = {epsilon_sq:.4f})')
ax2.set_ylim(bottom=0)

plt.tight_layout()
plt.show()



---
## Section 3: Argument Mining

Detect **claim** and **evidence** sentences using a two-stage pipeline:

1. **Stage 1 — binary detection:** `chkla/roberta-argument` flags each sentence as `ARGUMENT` or `NON-ARGUMENT`.
2. **Stage 2 — role assignment:** flagged sentences are mapped to `claim`, `evidence`, or `background` using the fine-tuned discourse classifier (v2). If Section 1 discourse labels are cached in the checkpoint (`sent_labels` column), the lookup is used directly — no additional GPU pass needed.

Features computed per essay:
- `n_claims` — count of claim sentences
- `n_evidence` — count of evidence sentences
- `arg_density` — (claims + evidence) / total sentences
- `claim_evidence_ratio` — claims / evidence (higher = more assertive, less supported)


In [ ]:

# Stage 1: chkla/roberta-argument — binary detector (Argument / NoArgument)
#          fine-tuned on the IBM Argument Quality corpus.
# Stage 2: fine-tuned discourse classifier → map Claim/Evidence labels directly.
arg_detector = pipeline(
    'text-classification',
    model='chkla/roberta-argument',
    device=0,
    truncation=True,
    max_length=512,
)

def classify_argumentative(sentences, detector=arg_detector,
                            ft_classifier=discourse_classifier):
    """
    Two-stage argument classification:
      1. arg_detector flags each sentence as Argument or NoArgument.
      2. Fine-tuned discourse classifier labels flagged sentences;
         Claim → 'claim', Evidence → 'evidence', anything else → 'background'.
    """
    if not sentences:
        return []
    detect_results = detector(sentences, batch_size=8)
    is_arg = [r['label'] == 'ARGUMENT' for r in detect_results]

    arg_sents = [s for s, flag in zip(sentences, is_arg) if flag]
    claim_labels = {}
    if arg_sents:
        ft_results = ft_classifier(arg_sents, batch_size=8)
        if isinstance(ft_results, dict):
            ft_results = [ft_results]
        for sent, res in zip(arg_sents, ft_results):
            lbl = res['label']
            claim_labels[sent] = 'claim' if lbl == 'Claim' else \
                                 'evidence' if lbl == 'Evidence' else 'background'

    return [claim_labels.get(s, 'background') for s in sentences]

# ── Sanity check: clearly argumentative sentences ─────────────────────────────
# These should all be flagged as ARGUMENT if the model is working correctly.
sanity_sents = [
    "Climate change is the defining crisis of our generation and governments must act immediately.",
    "Studies show that countries with universal healthcare have better population health outcomes.",
    "The death penalty should be abolished because it disproportionately affects minority communities.",
    "I love pizza.",   # control — clearly not argumentative
]
sanity_results = arg_detector(sanity_sents, batch_size=8)
print("── Sanity check ──────────────────────────────────────────────────────────")
for sent, res in zip(sanity_sents, sanity_results):
    print(f"  [{res['label']:>12s}  {res['score']:.2f}]  {sent}")

# ── Diagnostic demo on essay #2 — shows both pipeline stages ─────────────────
print("\n── Essay #2 demo ─────────────────────────────────────────────────────────")
demo_sents = split_sentences(df['essay'].iloc[1])
stage1 = arg_detector(demo_sents, batch_size=8)
is_arg  = [r['label'] == 'ARGUMENT' for r in stage1]

arg_sents = [s for s, flag in zip(demo_sents, is_arg) if flag]
if arg_sents:
    stage2 = discourse_classifier(arg_sents, batch_size=8)
    if isinstance(stage2, dict):
        stage2 = [stage2]
    stage2_map = {s: r['label'] for s, r in zip(arg_sents, stage2)}
else:
    stage2_map = {}

print(f"{'Sentence':<60} {'Stage1':>14} {'Stage2':>22} {'Final':>12}")
print('-' * 112)
for sent, arg_flag in zip(demo_sents, is_arg):
    s1 = 'ARGUMENT' if arg_flag else 'NON-ARGUMENT'
    s2 = stage2_map.get(sent, '—')
    if arg_flag:
        final = 'claim' if s2 == 'Claim' else 'evidence' if s2 == 'Evidence' else 'background'
    else:
        final = 'background'
    print(f"{sent[:58]:<60} {s1:>14} {s2:>22} {final:>12}")
print(f"\n{sum(is_arg)}/{len(demo_sents)} sentences flagged as ARGUMENT by Stage 1")


In [ ]:

import json

ARG_BATCH_SZ   = 64
ARG_CHUNK_SIZE = 500

# ── Check whether per-sentence discourse labels are available ──────────────────
# Section 1 now stores a 'sent_labels' JSON column in the discourse checkpoint.
# If present, we skip Stage 2 (re-running the discourse classifier) entirely and
# look up Claim/Evidence labels directly — faster and consistent with Section 1.
HAS_SENT_LABELS = 'sent_labels' in df.columns
if HAS_SENT_LABELS:
    # Build a lookup: student_id → {sentence_text: discourse_label}
    sent_label_lookup = {}
    for _, row in df.iterrows():
        try:
            sent_label_lookup[row['student_id']] = {
                item['sentence']: item['label']
                for item in json.loads(row['sent_labels'])
            }
        except (TypeError, json.JSONDecodeError):
            pass
    print(f'Using cached discourse labels from Section 1 for Stage 2 ({len(sent_label_lookup):,} essays).')
else:
    print('No cached discourse labels found — Stage 2 will re-run the discourse classifier.')

# ── Resume from checkpoint if it exists ───────────────────────────────────────
if ARG_CHECKPOINT_PATH.exists():
    saved_arg = pd.read_csv(ARG_CHECKPOINT_PATH)
    done_arg_ids = set(saved_arg['student_id'])
    arg_rows = saved_arg.to_dict('records')
    print(f'Resuming — {len(done_arg_ids):,} essays already processed.')
else:
    done_arg_ids = set()
    arg_rows = []

remaining_arg = df[~df['student_id'].isin(done_arg_ids)].reset_index(drop=True)

for chunk_start in tqdm(range(0, len(remaining_arg), ARG_CHUNK_SIZE), desc='Argument mining'):
    chunk = remaining_arg.iloc[chunk_start:chunk_start + ARG_CHUNK_SIZE]

    # Step 1: split sentences for this chunk
    chunk_sents = [split_sentences(row['essay']) for _, row in chunk.iterrows()]

    # Step 2: flatten and track per-essay boundaries
    flat_sents_arg, boundaries_arg = [], []
    for sents in chunk_sents:
        start = len(flat_sents_arg)
        flat_sents_arg.extend(sents)
        boundaries_arg.append((start, len(flat_sents_arg)))

    # Step 3: batched GPU pass — binary argument detection (single sentences, as trained)
    detect_results = []
    for i in range(0, len(flat_sents_arg), ARG_BATCH_SZ):
        batch = flat_sents_arg[i : i + ARG_BATCH_SZ]
        detect_results.extend(arg_detector(batch, batch_size=ARG_BATCH_SZ))

    is_arg_flat = [r['label'] == 'ARGUMENT' for r in detect_results]

    # Step 4: Claim/Evidence classification on flagged sentences.
    # Prefer cached Section 1 labels (no GPU call needed).
    # Fall back to re-running the discourse classifier if labels are unavailable.
    flagged_sents   = [s for s, flag in zip(flat_sents_arg, is_arg_flat) if flag]
    flagged_indices = [i for i, flag in enumerate(is_arg_flat) if flag]

    ft_label_map = {}   # flat index → 'claim' / 'evidence' / 'background'

    if HAS_SENT_LABELS:
        # Lookup from Section 1 discourse checkpoint — no model call
        for flat_i, sent in zip(flagged_indices, flagged_sents):
            essay_idx = next(
                ei for ei, (s, e) in enumerate(boundaries_arg) if s <= flat_i < e
            )
            sid = chunk.iloc[essay_idx]['student_id']
            lbl = sent_label_lookup.get(sid, {}).get(sent, '')
            ft_label_map[flat_i] = (
                'claim'    if lbl == 'Claim'    else
                'evidence' if lbl == 'Evidence' else
                'background'
            )
    elif flagged_sents:
        # Re-run discourse classifier (no cached labels available)
        ft_results = []
        for i in range(0, len(flagged_sents), ARG_BATCH_SZ):
            batch = flagged_sents[i : i + ARG_BATCH_SZ]
            ft_results.extend(discourse_classifier(batch, batch_size=ARG_BATCH_SZ))
        if isinstance(ft_results, dict):
            ft_results = [ft_results]
        for idx, res in zip(flagged_indices, ft_results):
            lbl = res['label']
            ft_label_map[idx] = (
                'claim'    if lbl == 'Claim'    else
                'evidence' if lbl == 'Evidence' else
                'background'
            )

    # Step 5: reconstruct per-essay features
    for (_, row), sents, (start, end) in zip(chunk.iterrows(), chunk_sents, boundaries_arg):
        labels     = [ft_label_map.get(i, 'background') for i in range(start, end)]
        n_claim    = labels.count('claim')
        n_evidence = labels.count('evidence')
        n_total    = len(sents)
        arg_rows.append({
            'student_id':           row['student_id'],
            'n_claims':             n_claim,
            'n_evidence':           n_evidence,
            'arg_density':          (n_claim + n_evidence) / max(n_total, 1),
            'claim_evidence_ratio': n_claim / max(n_evidence, 1),
        })

    # Step 6: save checkpoint
    pd.DataFrame(arg_rows).to_csv(ARG_CHECKPOINT_PATH, index=False)
    print(f'  Checkpoint saved — {len(arg_rows):,} essays total.')

arg_df = pd.DataFrame(arg_rows)
df = df.merge(arg_df, on='student_id')

corr_cols = ['n_claims', 'n_evidence', 'arg_density', 'claim_evidence_ratio', 'score']
print('Pearson correlations with score:')
print(df[corr_cols].corr()['score'].drop('score').round(3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, title in zip(
    axes,
    ['arg_density', 'claim_evidence_ratio'],
    ['Argument density vs score', 'Claim:Evidence ratio vs score'],
):
    ax.scatter(df[col], df['score'], alpha=0.4, edgecolors='none')
    m, b = np.polyfit(df[col].fillna(0), df['score'], 1)
    xs = np.linspace(df[col].min(), df[col].max(), 100)
    ax.plot(xs, m * xs + b, color='red', linewidth=1.5)
    ax.set_xlabel(col)
    ax.set_ylabel('Score')
    ax.set_title(title)
plt.tight_layout()
plt.show()

---
## Section 4: Coherence

Measure **local coherence** as the mean cosine similarity between consecutive
sentence embeddings. High scores indicate that each sentence stays semantically
close to its neighbour — a proxy for topical focus and smooth transitions.

We also compute a **coherence arc** (same 10-bucket normalisation as sentiment)
to see whether coherence dips or peaks at particular points in the essay.

In [ ]:

embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def local_coherence(sentences, model=embedder):
    """Mean cosine similarity between adjacent sentence embeddings."""
    if len(sentences) < 2:
        return float('nan')
    embs = model.encode(sentences, show_progress_bar=False, normalize_embeddings=True)
    return float(np.mean([
        cosine_similarity([embs[i]], [embs[i + 1]])[0][0]
        for i in range(len(embs) - 1)
    ]))

def coherence_profile(sentences, model=embedder, n_buckets=ARC_BUCKETS):
    """Coherence score resampled to n_buckets for arc-style analysis."""
    if len(sentences) < 2:
        return [float('nan')] * n_buckets
    embs = model.encode(sentences, show_progress_bar=False, normalize_embeddings=True)
    sims = [cosine_similarity([embs[i]], [embs[i + 1]])[0][0]
            for i in range(len(embs) - 1)]
    x_orig = np.linspace(0, 1, len(sims))
    x_new  = np.linspace(0, 1, n_buckets)
    return list(np.interp(x_new, x_orig, sims))

COH_CHUNK_SIZE = 500

# ── Resume from checkpoint if it exists ───────────────────────────────────────
if COH_CHECKPOINT_PATH.exists():
    saved_coh = pd.read_csv(COH_CHECKPOINT_PATH)
    done_coh_ids = set(saved_coh['student_id'])
    coherence_rows = saved_coh.to_dict('records')
    print(f'Resuming — {len(done_coh_ids):,} essays already processed.')
else:
    done_coh_ids = set()
    coherence_rows = []

remaining_coh = df[~df['student_id'].isin(done_coh_ids)].reset_index(drop=True)

# ── Process in chunks, saving every COH_CHUNK_SIZE essays ─────────────────────
for chunk_start in tqdm(range(0, len(remaining_coh), COH_CHUNK_SIZE), desc='Coherence'):
    chunk = remaining_coh.iloc[chunk_start:chunk_start + COH_CHUNK_SIZE]

    for _, row in chunk.iterrows():
        sents   = split_sentences(row['essay'])
        coh_sc  = local_coherence(sents)
        profile = coherence_profile(sents)
        coherence_rows.append({
            'student_id': row['student_id'],
            'coherence_score': coh_sc,
            **{f'coh_{i}': profile[i] for i in range(ARC_BUCKETS)},
        })

    pd.DataFrame(coherence_rows).to_csv(COH_CHECKPOINT_PATH, index=False)
    print(f'  Checkpoint saved — {len(coherence_rows):,} essays total.')

coh_df = pd.DataFrame(coherence_rows)
df = df.merge(coh_df, on='student_id')

print(f"Coherence — mean: {df['coherence_score'].mean():.3f}, "
      f"std: {df['coherence_score'].std():.3f}")
print(f"Correlation with score: {df['coherence_score'].corr(df['score']):.3f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Scatter: coherence vs reviewer score
ax = axes[0]
valid = df.dropna(subset=['coherence_score'])
ax.scatter(valid['coherence_score'], valid['score'], alpha=0.4, edgecolors='none')
m, b = np.polyfit(valid['coherence_score'], valid['score'], 1)
xs = np.linspace(valid['coherence_score'].min(), valid['coherence_score'].max(), 100)
ax.plot(xs, m * xs + b, color='red', linewidth=1.5)
ax.set_xlabel('Local coherence score')
ax.set_ylabel('Reviewer score')
ax.set_title('Coherence vs score')

# Mean coherence profile across the corpus
ax2 = axes[1]
coh_cols     = [f'coh_{i}' for i in range(ARC_BUCKETS)]
mean_profile = df[coh_cols].mean().values
ax2.plot(mean_profile, marker='o', linewidth=2)
ax2.set_xticks(range(ARC_BUCKETS))
ax2.set_xticklabels([f'{int(i/ARC_BUCKETS*100)}%' for i in range(ARC_BUCKETS)])
ax2.set_xlabel('Essay position')
ax2.set_ylabel('Mean cosine similarity')
ax2.set_title('Coherence profile across corpus')

plt.tight_layout()
plt.show()

---
## Section 5: Combined feature summary & export

In [ ]:
# Metadata columns present in synthetic/freeform CSVs — include them if available
# so outputs don't require a rejoin on student_id to recover model/persona/etc.
METADATA_COLS = ['model', 'topic', 'register', 'persona_key',
                 'replication', 'target_words', 'target_sentences']
meta_in_df = [c for c in METADATA_COLS if c in df.columns]

summary_cols = [
    'student_id', 'score',
    *meta_in_df,
    'discourse_cluster', 'arc_label',
    'n_claims', 'n_evidence', 'arg_density', 'claim_evidence_ratio',
    'coherence_score',
]
summary = df[summary_cols].copy()
print(summary.describe().round(3))
summary.head()


In [ ]:

# ── Sentence & word count distributions (real corpus) ─────────────────────────
# Both are exported for use in generate_synthetic_essays.ipynb.
#
# NOTE: The prompt imposes a hard 650-word cap. The word count distribution is
# left-skewed with a hard ceiling — no essay exceeds 650 words, and the top
# decile piles up at exactly 650. The mean (590) is pulled down by the ~35% of
# essays that are under-cap; the median (627) is a more representative target.
n_sent  = df['n_sentences'].dropna()
n_words = df['essay'].dropna().str.split().str.len()

WORD_CAP = 650   # hard prompt ceiling
pctls = [10, 25, 50, 75, 90]

print(f"Real corpus — {n_words.count():,} essays")
print()
print("Sentences per essay:")
print(f"  mean: {n_sent.mean():.1f}  |  std: {n_sent.std():.1f}  |  median: {n_sent.median():.0f}")
for p in pctls:
    print(f"  p{p:2d}: {np.percentile(n_sent, p):.0f}")

print()
print("Words per essay (hard cap = 650):")
print(f"  mean: {n_words.mean():.0f}  |  std: {n_words.std():.0f}  |  median: {n_words.median():.0f}")
for p in pctls:
    print(f"  p{p:2d}: {np.percentile(n_words, p):.0f}")
print(f"  At cap (650 words): {(n_words >= WORD_CAP).sum():,}  ({(n_words >= WORD_CAP).mean()*100:.1f}%)")
print(f"  Under-cap (<650):   {(n_words < WORD_CAP).sum():,}  ({(n_words < WORD_CAP).mean()*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))

# Left: sentence count
ax = axes[0]
ax.hist(n_sent, bins=40, edgecolor='white', linewidth=0.4)
ax.axvline(n_sent.mean(),              color='red',    linewidth=1.5, linestyle='--',
           label=f"mean ({n_sent.mean():.1f})")
ax.axvline(n_sent.median(),            color='purple', linewidth=1.5, linestyle='-.',
           label=f"median ({n_sent.median():.0f})")
ax.axvline(np.percentile(n_sent, 10), color='orange', linewidth=1.2, linestyle=':',
           label=f"p10 ({np.percentile(n_sent, 10):.0f})")
ax.axvline(np.percentile(n_sent, 90), color='green',  linewidth=1.2, linestyle=':',
           label=f"p90 ({np.percentile(n_sent, 90):.0f})")
ax.set_xlabel('Sentences per essay')
ax.set_ylabel('Count')
ax.set_title('Sentence-count distribution')
ax.legend(fontsize=8)

# Right: word count — extend x-axis slightly past 650 so the cap bar is visible
ax2 = axes[1]
bin_edges = range(200, 680, 10)
ax2.hist(n_words, bins=bin_edges, edgecolor='white', linewidth=0.4, color='steelblue')
# Shade the cap region
ax2.axvspan(WORD_CAP - 10, WORD_CAP + 5, color='red', alpha=0.12,
            label=f'650-word cap ({(n_words >= WORD_CAP).mean()*100:.0f}% at cap)')
ax2.axvline(n_words.mean(),              color='red',    linewidth=1.5, linestyle='--',
            label=f"mean ({n_words.mean():.0f})")
ax2.axvline(n_words.median(),            color='purple', linewidth=1.5, linestyle='-.',
            label=f"median ({n_words.median():.0f})")
ax2.axvline(np.percentile(n_words, 10), color='orange', linewidth=1.2, linestyle=':',
            label=f"p10 ({np.percentile(n_words, 10):.0f})")
ax2.set_xlabel('Words per essay')
ax2.set_ylabel('Count')
ax2.set_title('Word-count distribution  (hard cap = 650)')
ax2.set_xlim(200, 670)
ax2.legend(fontsize=8)

plt.suptitle('Real essay corpus — length distributions', y=1.02)
plt.tight_layout()
plt.show()

# ── Export constants for generate_synthetic_essays.ipynb ─────────────────────
# Use median instead of mean for word count: the mean is downward-biased by
# the censored left tail. The median (627) better reflects what the LLM should aim
# for. For sentence count there is no hard cap, so mean is appropriate.
N_SENT_MEAN = float(n_sent.mean())
N_SENT_STD  = float(n_sent.std())
N_SENT_P10  = float(np.percentile(n_sent, 10))
N_SENT_P90  = float(np.percentile(n_sent, 90))

N_WORD_MEDIAN = float(n_words.median())   # preferred target — unbiased by cap
N_WORD_MEAN   = float(n_words.mean())     # kept for reference
N_WORD_STD    = float(n_words.std())
N_WORD_P10    = float(np.percentile(n_words, 10))
N_WORD_P90    = float(np.percentile(n_words, 90))
N_WORD_CAP    = WORD_CAP

print(f"\nSentence target: N({N_SENT_MEAN:.1f}, {N_SENT_STD:.1f}), "
      f"clip [{N_SENT_P10:.0f}, {N_SENT_P90:.0f}]")
print(f"Word target:     median = {N_WORD_MEDIAN:.0f}, "
      f"p10–p90 = [{N_WORD_P10:.0f}, {N_WORD_P90:.0f}], hard cap = {N_WORD_CAP}")
print(f"\nMean words/sentence: {N_WORD_MEAN / N_SENT_MEAN:.1f}  "
      f"(use word count as primary LLM target; sentence count as secondary)")
print(f"Recommended prompt phrasing: "
      f"'Write approximately {int(N_WORD_MEDIAN)} words (about {int(N_SENT_MEAN)} sentences), "
      f"not exceeding {N_WORD_CAP} words.'")


In [ ]:

# ── 1. Full numeric feature correlation matrix (individual essays) ─────────────
arc_bucket_cols = [f'arc_{i}' for i in range(ARC_BUCKETS)]

feature_cols = [
    'n_claims', 'n_evidence', 'arg_density', 'claim_evidence_ratio',
    'coherence_score',
    *arc_bucket_cols,
    'arc_cluster',
    'score',
]

# One-hot encode categorical metadata columns when present (synthetic/freeform).
# This lets the heatmap show e.g. whether gpt-4o essays cluster into particular
# arc shapes, or whether 'strong' register correlates with higher coherence.
CATEGORICAL_META = ['model', 'topic_key', 'register_key', 'persona_key']
cat_in_df = [c for c in CATEGORICAL_META if c in df.columns]

corr_df = df[feature_cols].copy()
if cat_in_df:
    dummies = pd.get_dummies(df[cat_in_df], prefix_sep='=').astype(float)
    corr_df = pd.concat([corr_df, dummies], axis=1)
    print(f"Added one-hot columns for: {cat_in_df}  ({dummies.shape[1]} dummies total)")

corr_matrix = corr_df.corr()

fig_h = max(12, len(corr_df.columns) * 0.55)
fig_w = max(14, len(corr_df.columns) * 0.6)
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, square=True, linewidths=0.4, annot_kws={'size': 6})
ax.set_title(
    'Feature correlation matrix — individual essays\n'
    '(arc_0…arc_9 = valence at each 10% position'
    + ('; one-hot metadata appended' if cat_in_df else '') + ')'
)
plt.tight_layout()
plt.show()

# ── 2. Score correlations only — sorted, easier to read ───────────────────────
score_corr = corr_matrix['score'].drop('score').sort_values(key=abs, ascending=False)
print("Correlations with score (|r| descending):")
print(score_corr.round(3).to_string())

# ── 3. Cluster-mean feature profile heatmap ───────────────────────────────────
cluster_mean_cols = ['score', 'coherence_score', 'n_claims', 'n_evidence',
                     'arg_density', 'claim_evidence_ratio', *arc_bucket_cols]
cluster_means = df.groupby('arc_cluster')[cluster_mean_cols].mean()
cluster_means.index = [arc_names.get(i, str(i)) for i in cluster_means.index]

cluster_means_z = (cluster_means - cluster_means.mean()) / cluster_means.std()

fig2, ax2 = plt.subplots(figsize=(16, 5))
sns.heatmap(cluster_means_z, annot=cluster_means.round(2), fmt='',
            cmap='coolwarm', center=0, ax=ax2, linewidths=0.4,
            annot_kws={'size': 7})
ax2.set_title('Arc cluster feature profiles (cell = raw mean; colour = z-score across clusters)')
ax2.set_xlabel('Feature')
ax2.set_ylabel('Arc cluster')
plt.tight_layout()
plt.show()

print("\nCluster-mean feature summary (sorted by mean score):")
print(cluster_means.sort_values('score').round(3).to_string())

# ── 4. Metadata × arc cluster breakdown (synthetic/freeform only) ─────────────
if cat_in_df:
    for col in cat_in_df:
        ct = pd.crosstab(df[col], df['arc_label'], normalize='index').round(3)
        print(f"\nArc label distribution by {col} (row-normalised):")
        with pd.option_context('display.max_columns', None, 'display.width', None):
            print(ct.to_string())


In [ ]:

# ── Multi-way metadata breakdown ──────────────────────────────────────────────
# For each metadata pair produces three blocks:
#
#   A. Categorical outcomes (arc_label, discourse_cluster)
#      - Proportion heatmap: rows = (col_a, col_b), cols = outcome categories
#      - Faceted stacked-bar: one panel per col_a value
#
#   B. Numeric outcomes (arg_density, n_claims, n_evidence,
#                        claim_evidence_ratio, coherence_score)
#      - Grid of pivot heatmaps: rows = col_a, cols = col_b, cell = mean value

META_PAIRS = [
    ('model', 'persona_key'),
    ('model', 'register_key'),
    ('register_key', 'persona_key'),
    ('topic_key', 'model'),
    ('topic_key', 'register_key'),
]
available_pairs = [(a, b) for a, b in META_PAIRS if a in df.columns and b in df.columns]

CATEGORICAL_OUTCOMES = {
    'arc_label':         'Arc label',
    'discourse_cluster': 'Discourse cluster',
}
NUMERIC_OUTCOMES = {
    'arg_density':          'Arg density',
    'n_claims':             'N claims',
    'n_evidence':           'N evidence',
    'claim_evidence_ratio': 'Claim:Evidence ratio',
    'coherence_score':      'Coherence score',
}

if not available_pairs:
    print("No metadata pairs found in df -- skipping multi-way feature breakdown.")
else:
    for col_a, col_b in available_pairs:
        print(f"\n{'='*70}")
        print(f"  {col_a}  x  {col_b}")
        print(f"{'='*70}")

        # ── A. Categorical outcomes ────────────────────────────────────────────
        for outcome_col, outcome_label in CATEGORICAL_OUTCOMES.items():
            if outcome_col not in df.columns:
                continue

            all_labels = sorted(df[outcome_col].unique(), key=str)
            _oc = plt.cm.tab10(np.linspace(0, 0.9, len(all_labels)))
            cat_palette = dict(zip(all_labels, _oc))

            # Proportion heatmap
            ct = pd.crosstab(
                [df[col_a], df[col_b]], df[outcome_col], normalize='index'
            ).round(3)
            ct = ct.reindex(columns=all_labels, fill_value=0)

            fig_h = max(5, len(ct) * 0.5)
            fig_w = max(8, len(ct.columns) * 1.4)
            fig, ax = plt.subplots(figsize=(fig_w, fig_h))
            sns.heatmap(
                ct, annot=True, fmt='.2f', cmap='YlOrRd',
                ax=ax, linewidths=0.4, annot_kws={'size': 8},
                vmin=0, vmax=max(ct.values.max(), 0.01),
            )
            ax.set_title(f'{outcome_label} -- {col_a} x {col_b}  (row-normalised proportion)')
            ax.set_xlabel(outcome_label)
            ax.set_ylabel(f'{col_a}  /  {col_b}')
            plt.tight_layout()
            plt.show()

            # Faceted stacked-bar: one panel per col_a value
            vals_a   = sorted(df[col_a].unique())
            n_panels = len(vals_a)
            fig, axes = plt.subplots(
                1, n_panels,
                figsize=(max(4, 3.2 * n_panels), 5),
                sharey=True,
            )
            if n_panels == 1:
                axes = [axes]

            for ax, val_a in zip(axes, vals_a):
                sub    = df[df[col_a] == val_a]
                ct_sub = pd.crosstab(sub[col_b], sub[outcome_col], normalize='index')
                ct_sub = ct_sub.reindex(columns=all_labels, fill_value=0)
                x      = np.arange(len(ct_sub))
                bottom = np.zeros(len(ct_sub))
                for lbl in all_labels:
                    heights = ct_sub[lbl].values
                    ax.bar(x, heights, bottom=bottom,
                           color=cat_palette[lbl], width=0.65, label=str(lbl))
                    bottom += heights
                ax.set_xticks(x)
                ax.set_xticklabels(ct_sub.index, rotation=35, ha='right', fontsize=8)
                ax.set_title(str(val_a), fontsize=9)
                ax.set_ylim(0, 1.05)
                if ax is axes[0]:
                    ax.set_ylabel('Proportion')

            handles = [plt.Rectangle((0, 0), 1, 1, color=cat_palette[l]) for l in all_labels]
            axes[-1].legend(
                handles, [str(l) for l in all_labels],
                bbox_to_anchor=(1.05, 1), loc='upper left',
                fontsize=7, title=outcome_label,
            )
            fig.suptitle(
                f'{outcome_label} distribution by {col_b},  faceted by {col_a}',
                y=1.02, fontsize=11,
            )
            plt.tight_layout()
            plt.show()

        # ── B. Numeric outcomes: pivot heatmap grid ────────────────────────────
        numeric_present = [c for c in NUMERIC_OUTCOMES if c in df.columns]
        if numeric_present:
            n_feats = len(numeric_present)
            vals_a  = sorted(df[col_a].unique())
            vals_b  = sorted(df[col_b].unique())

            fig, axes = plt.subplots(
                1, n_feats,
                figsize=(max(5, 4.5 * n_feats), max(3, 0.7 * len(vals_a) + 2)),
                squeeze=False,
            )
            for c_idx, feat in enumerate(numeric_present):
                feat_label = NUMERIC_OUTCOMES[feat]
                ax = axes[0][c_idx]
                pivot = (
                    df.groupby([col_a, col_b])[feat]
                    .mean()
                    .unstack(col_b)
                    .reindex(index=vals_a, columns=vals_b)
                )
                sns.heatmap(
                    pivot, annot=True, fmt='.2f', cmap='coolwarm',
                    center=pivot.stack().median(),
                    ax=ax, linewidths=0.4, annot_kws={'size': 8},
                    cbar_kws={'shrink': 0.7},
                )
                ax.set_title(feat_label, fontsize=9)
                ax.set_xlabel(col_b, fontsize=8)
                ax.set_ylabel(col_a if c_idx == 0 else '', fontsize=8)
                ax.tick_params(axis='x', rotation=35, labelsize=7)
                ax.tick_params(axis='y', rotation=0,  labelsize=7)

            fig.suptitle(
                f'Mean numeric features -- {col_a} (rows) x {col_b} (cols)',
                y=1.02, fontsize=11,
            )
            plt.tight_layout()
            plt.show()


# ── Standalone topic_key breakdown ────────────────────────────────────────────
# Shows the independent effect of prompt topic on every outcome, without
# needing to facet by a second variable.  Answers: "how much was the
# prompt type driving these results?"
if 'topic_key' in df.columns:
    print(f"\n{'='*70}")
    print("  topic_key  (standalone -- independent prompt effect)")
    print(f"{'='*70}")

    topic_vals = sorted(df['topic_key'].unique())
    x_pos_t    = np.arange(len(topic_vals))

    # A. Categorical outcomes: one stacked-bar chart per outcome
    for outcome_col, outcome_label in CATEGORICAL_OUTCOMES.items():
        if outcome_col not in df.columns:
            continue
        all_labels_t = sorted(df[outcome_col].unique(), key=str)
        _oc_t        = plt.cm.tab10(np.linspace(0, 0.9, len(all_labels_t)))
        pal_t        = dict(zip(all_labels_t, _oc_t))

        ct_t = pd.crosstab(df['topic_key'], df[outcome_col], normalize='index')
        ct_t = ct_t.reindex(index=topic_vals, columns=all_labels_t, fill_value=0)

        fig, ax = plt.subplots(figsize=(max(8, len(topic_vals) * 1.1), 5))
        bottom_t = np.zeros(len(topic_vals))
        for lbl_t in all_labels_t:
            heights_t = ct_t[lbl_t].values
            ax.bar(x_pos_t, heights_t, bottom=bottom_t,
                   color=pal_t[lbl_t], width=0.7, label=str(lbl_t))
            bottom_t += heights_t
        ax.set_xticks(x_pos_t)
        ax.set_xticklabels(topic_vals, rotation=40, ha='right', fontsize=9)
        ax.set_ylabel('Proportion')
        ax.set_ylim(0, 1.05)
        ax.set_title(f'{outcome_label} distribution by topic_key (row-normalised)')
        ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8,
                  title=outcome_label)
        plt.tight_layout()
        plt.show()

    # B. Numeric outcomes: bar charts (mean +/- SE) per topic, one subplot each
    numeric_t = [c for c in NUMERIC_OUTCOMES if c in df.columns]
    if numeric_t:
        n_feats_t   = len(numeric_t)
        topic_stats = df.groupby('topic_key')[numeric_t].agg(['mean', 'sem'])

        fig, axes_t = plt.subplots(
            1, n_feats_t,
            figsize=(max(5, 4.5 * n_feats_t), 4.5),
            squeeze=False,
        )
        for c_idx_t, feat_t in enumerate(numeric_t):
            ax_t    = axes_t[0][c_idx_t]
            means_t = topic_stats[(feat_t, 'mean')].reindex(topic_vals)
            sems_t  = topic_stats[(feat_t, 'sem')].reindex(topic_vals)
            ax_t.bar(x_pos_t, means_t, yerr=sems_t, capsize=4,
                     color='steelblue', alpha=0.8, edgecolor='grey')
            ax_t.set_xticks(x_pos_t)
            ax_t.set_xticklabels(topic_vals, rotation=40, ha='right', fontsize=8)
            ax_t.set_title(NUMERIC_OUTCOMES[feat_t], fontsize=9)
            ax_t.set_ylabel('Mean' if c_idx_t == 0 else '')

        fig.suptitle('Mean numeric features by topic_key  (error bars = SE)',
                     y=1.02, fontsize=11)
        plt.tight_layout()
        plt.show()


In [ ]:
output_dir = OUTPUT_DIR
output_dir.mkdir(parents=True, exist_ok=True)

summary.to_csv(output_dir / 'discourse_features.csv', index=False)

arc_extra  = [f'arc_{i}' for i in range(ARC_BUCKETS)]
coh_extra  = [f'coh_{i}' for i in range(ARC_BUCKETS)]
df[summary_cols + arc_extra + coh_extra].to_csv(
    output_dir / 'full_features.csv', index=False
)
print('Saved to:', output_dir)


## Section 6: Real vs. Synthetic Discriminability

Compares discourse features between real human essays and AI-generated ones.
Only runs when `DATASET != 'real'` (requires the synthetic or freeform full_features.csv to exist).
Results inform feature weights in `ai_detection/ai_detection.ipynb`.

In [ ]:
from scipy import stats
from statsmodels.stats.multitest import multipletests

# Only run when the synthetic (or freeform) export has been produced
REAL_DISC_PATH = Path('../outputs/discourse/real/full_features.csv')

if DATASET == 'real':
    print("Section 6 skipped — set DATASET = 'synthetic' or 'freeform' to run discriminability analysis.")
elif not REAL_DISC_PATH.exists():
    print(f"Section 6 skipped — real discourse features not found at: {REAL_DISC_PATH}")
else:
    real_disc = pd.read_csv(REAL_DISC_PATH)
    syn_disc  = df.copy()   # current run is already the AI corpus

    # Scalar discourse features to compare (exclude arc/coherence bucket arrays)
    disc_feats = ['n_claims', 'n_evidence', 'arg_density', 'claim_evidence_ratio', 'coherence_score']
    disc_feats = [f for f in disc_feats if f in real_disc.columns and f in syn_disc.columns]

    def rank_biserial(u_stat, n1, n2):
        return 1 - (2 * u_stat) / (n1 * n2)

    rows = []
    for feat in disc_feats:
        a = real_disc[feat].dropna()
        b = syn_disc[feat].dropna()
        u, p = stats.mannwhitneyu(a, b, alternative='two-sided')
        r = rank_biserial(u, len(a), len(b))
        rows.append({'feature': feat, 'r_real_vs_ai': r, 'p': p,
                     'mean_real': a.mean(), 'mean_ai': b.mean()})

    eff = pd.DataFrame(rows)
    _, pvals_bh, _, _ = multipletests(eff['p'], method='fdr_bh')
    eff['p_bh'] = pvals_bh
    eff['abs_r'] = eff['r_real_vs_ai'].abs()
    eff = eff.sort_values('abs_r', ascending=False).reset_index(drop=True)

    print(f'Discourse discriminability — real ({len(real_disc):,}) vs. {DATASET} ({len(syn_disc):,}):')
    print(eff[['feature', 'mean_real', 'mean_ai', 'r_real_vs_ai', 'p_bh']].round(3).to_string(index=False))

In [ ]:
if DATASET != 'real' and REAL_DISC_PATH.exists():
    import matplotlib.patches as mpatches

    # ── Scatter: coherence vs. arg_density ───────────────────────────────────
    if 'coherence_score' in disc_feats and 'arg_density' in disc_feats:
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.scatter(real_disc['arg_density'], real_disc['coherence_score'],
                   alpha=0.15, s=10, color='#4C72B0', label='real')
        ax.scatter(syn_disc['arg_density'], syn_disc['coherence_score'],
                   alpha=0.4, s=20, color='#DD8452', label=DATASET)
        ax.set_xlabel('arg_density')
        ax.set_ylabel('coherence_score')
        ax.set_title(f'Discourse space: real vs. {DATASET}')
        ax.legend()
        plt.tight_layout()
        plt.show()

    # ── Discourse cluster composition ─────────────────────────────────────────
    if 'discourse_cluster' in real_disc.columns and 'discourse_cluster' in syn_disc.columns:
        real_disc['_corpus'] = 'real'
        syn_disc['_corpus']  = DATASET
        combined = pd.concat([real_disc[['discourse_cluster', '_corpus']],
                              syn_disc[['discourse_cluster', '_corpus']]])
        cluster_counts = combined.groupby(['discourse_cluster', '_corpus']).size().unstack(fill_value=0)
        cluster_pct = cluster_counts.div(cluster_counts.sum(axis=1), axis=0)

        cluster_pct.plot(kind='bar', figsize=(8, 5), color=['#4C72B0', '#DD8452'])
        plt.title(f'Discourse cluster composition: real vs. {DATASET}')
        plt.xlabel('Discourse cluster')
        plt.ylabel('Fraction of essays')
        plt.xticks(rotation=0)
        plt.legend(title='corpus')
        plt.tight_layout()
        plt.show()